In [ ]:
#Notebook formatting for Jupyter.
from IPython.display import display, HTML
display(HTML("<style>.jp-Cell { margin-left: -50% !important; margin-right: -50% !important; }</style>"))

In [ ]:
import sys
print("Python executable:", sys.executable)
print("udemy_env kernel is now active!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import gc
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow as pa
print('complete')

In [ ]:
sns.set_style("whitegrid")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
%load_ext memory_profiler

In [ ]:
%load_ext autotime

In [ ]:
#Note - DFR load attempts that wont work:
#dfr_data = pq.read_table('LCR_cleaned_final.parquet', nrows=1_000_000)
#dfr = dfr_data.to_pandas().sample(100000, random_state=42)
#dfr = pd.read_parquet('LCR_cleaned_final.parquet')
#dfr_sample = dfr.sample(n=100000, random_state=42)

#dfr_data = pq.ParquetFile('LCR_cleaned_final.parquet')
#batch = next(pf.iter_batches(batch_size=1_000_000))
#dfr = pa.Table.from_batches([batch]).to_pandas().sample(100000, random_state=42)

#dataset = pq.ParquetDataset('LCR_cleaned_final')
#table = dataset.read_pandas(use_threads=True)
#dfr = table.to_pandas().head(1000000)

In [ ]:
# This cell tests whether our dfr sample is representative of the full dataset.
# Run this cell first, compare the .describe() outputs, then comment out this cell and the cell underneath once a good sample size is confirmed.

# Reason: The full LCR_cleaned_final dataset is very large (~9GB, 27.6M rows). I used a smaller random sample for faster analysis and statistical representativeness.

risk_score_pop = pd.read_parquet('LCR_cleaned_final', columns=['risk_score', 'dti'])
risk_score_sample = risk_score_pop.sample(500000, random_state=42)
print('population description')
print(risk_score_pop.describe())
print(f"Row Count: {len(risk_score_pop)}")
print(f"   risk_score_pop memory usage: {risk_score_pop.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

In [ ]:
print('sample description')
print(risk_score_sample.describe())
del risk_score_pop
del risk_score_sample

In [ ]:
print("Loading important columns for quick testing.")
cols = [
    'addr_state',
'annual_inc',
'delinq_2yrs',
'dti',
'emp_length_lt1',
'emp_length',
'fico_range_high',
'fico_range_low',
'home_ownership',
'inq_last_6mths',
'int_rate',
'issue_d',
'last_fico_range_high',
'last_fico_range_low',
'loan_amnt',
'loan_status',
'pub_rec',
'purpose',
'revol_bal',
'revol_util',
'sub_grade',
'term',
'verification_status',
]
dfa_data = pd.read_parquet('LCA_cleaned_final', columns=cols) #2,260,701
dfa = pd.read_parquet('LCA_cleaned_final', columns=cols).head(500000)  
#dfa = dfa_data.sample(100000, random_state=42)


dfr_data = ds.dataset('LCR_cleaned_final').scanner().head(1000).to_pandas() #Currently at 1k for basic analysis and saving RAM. Set to 500k for a correctly representative sample. 
#dfr = dfr_data.sample(500000, random_state=42)


#test = pd.read_parquet('LCR_cleaned_final') #27,648,741 rows. Appx ~9GB
print("loaded")

### Quick exploration

In [ ]:
print(f"Loaded dfa: {dfa_data.shape[0]:,} rows, {dfa_data.shape[1]} columns")
print("Initial memory usage:")
print(f"   dfa: {dfa_data.memory_usage(deep=True).sum() / (1024**3):.1f} GB")
print(f"   dfr: {dfr_data.memory_usage(deep=True).sum() / (1024**2):.1f} MB")

In [ ]:
dfa_data.head(5)

In [ ]:
dfr_data.head(5)

In [ ]:
dfa_data.info()

In [ ]:
dfr_data.info()

In [ ]:
dfa.describe()

In [ ]:
dfa_data.describe().round(2)

In [ ]:
dfr_data.describe().round(2)

### Creating cohort framework and more testing

In [ ]:
status_cts = dfa_data['loan_status'].value_counts()
status_p = dfa_data['loan_status'].value_counts(normalize=True)
by_year_cts = dfa_data['issue_d'].dt.year.value_counts()
by_year_p = dfa_data['issue_d'].dt.year.value_counts(normalize=True)
by_purpose_cts = dfa_data['purpose'].value_counts()
by_purpose_p = dfa_data['purpose'].value_counts(normalize=True)
by_state_cts = dfa_data['addr_state'].value_counts()
by_state_p = dfa_data['addr_state'].value_counts(normalize=True)
by_grade_cts = dfa_data['sub_grade'].value_counts()
by_grade_p = dfa_data['sub_grade'].value_counts(normalize=True)

dlqcy = ['Charged Off','Late (31-120 days)','Late (16-30 days)','Default']
delinquencies = dfa_data['loan_status'].isin(dlqcy).value_counts()
delinquencies_p = dfa_data['loan_status'].isin(dlqcy).value_counts(normalize=True)
delinquencies_df = dfa_data[dfa_data['loan_status'].isin(dlqcy)]

dfa_data['delinquency_tf'] = dfa_data['loan_status'].isin(dlqcy).astype(int)
dflt = ['Charged Off','Default']
dfa_data['defaulted_tf'] = dfa_data['loan_status'].isin(dflt).astype(int)


dfa['defaulted_tf'] = dfa['loan_status'].isin(dflt).astype(int)
dfa['delinquency_tf'] = dfa['loan_status'].isin(dlqcy).astype(int)

In [ ]:
delinquency_by_subgrade = dfa_data.groupby('sub_grade')['delinquency_tf'].value_counts()

In [ ]:
#Creating delinquency rate/proportion cohorts by:
# Sub grade
# Issuance date
# Purpose
# State
# Employment length of one year or less.
# Employment Length
# Debt-to-Income ratio


delinquency_rate_subgrade = dfa_data.groupby('sub_grade')['delinquency_tf'].mean() 
delinquency_rate_year = dfa_data.groupby(dfa_data.issue_d.dt.year)['delinquency_tf'].mean() 
delinquency_rate_purpose = dfa_data.groupby('purpose')['delinquency_tf'].mean() 
delinquency_rate_state = dfa_data.groupby('addr_state')['delinquency_tf'].mean() 
delinquency_rate_low_employment = dfa_data.groupby('emp_length_lt1')['delinquency_tf'].mean() 
delinquency_rate_emp_length = dfa_data.groupby('emp_length')['delinquency_tf'].mean() 
delinquency_rate_dti = dfa_data.groupby(pd.cut(dfa_data['dti'], bins=10), observed=True)['delinquency_tf'].mean()

default_rate_subgrade = dfa_data.groupby('sub_grade')['defaulted_tf'].mean() 
default_rate_year = dfa_data.groupby(dfa_data.issue_d.dt.year)['defaulted_tf'].mean() 
default_rate_purpose = dfa_data.groupby('purpose')['defaulted_tf'].mean() 
default_rate_state = dfa_data.groupby('addr_state')['defaulted_tf'].mean() 
default_rate_low_employment = dfa_data.groupby('emp_length_lt1')['defaulted_tf'].mean() 
default_rate_emp_length = dfa_data.groupby('emp_length')['defaulted_tf'].mean() 
default_rate_dti = dfa_data.groupby(pd.cut(dfa_data['dti'], bins=10), observed=True)['defaulted_tf'].mean()

In [ ]:
print(delinquency_rate_dti, default_rate_dti)

In [ ]:
delinquency_rate_year

In [ ]:
default_rate_year

In [ ]:
dfa_data.groupby(dfa_data.issue_d.dt.year)['delinquency_tf'].value_counts() 

In [ ]:
year_summary = dfa_data.groupby(dfa_data['issue_d'].dt.year).agg(
    total_loans=('loan_status', 'count'),
    delinquencies=('delinquency_tf', 'sum'),
    defaults=('defaulted_tf', 'sum')
)

year_summary['delinquency_rate'] = year_summary['delinquencies'] / year_summary['total_loans']
year_summary['default_rate'] = year_summary['defaults'] / year_summary['total_loans']

year_summary['delinquency_rate'] = (year_summary['delinquency_rate'] * 100).round(2)
year_summary['default_rate'] = (year_summary['default_rate'] * 100).round(2)

print(year_summary)

In [ ]:
year_summary['delinquencies'] - year_summary['defaults']

In [ ]:
status_cts

In [ ]:
loan_status_by_year = dfa_data.groupby([dfa_data['issue_d'].dt.year, 'loan_status']).size()

print(loan_status_by_year)

In [ ]:
vintage = dfa_data.groupby(dfa_data['issue_d'].dt.year).agg(
    total_loans=('loan_status', 'count'),
    charged_off=('loan_status', lambda x: (x == 'Charged Off').sum()),
    default=('loan_status', lambda x: (x == 'Default').sum()),
    late_31_120=('loan_status', lambda x: (x == 'Late (31-120 days)').sum()),
    fully_paid=('loan_status', lambda x: (x == 'Fully Paid').sum()),
    current=('loan_status', lambda x: (x == 'Current').sum())
)

vintage['delinquency_rate'] = (vintage['charged_off'] + vintage['late_31_120']) / vintage['total_loans'] * 100
vintage['default_rate'] = vintage['charged_off'] / vintage['total_loans'] * 100   # Most common definition

vintage = vintage.round(2)
print(vintage)

### Quick reflection:

- Lending Club experienced explosive growth in loan volume starting in 2013, peaking in 2017–2018.
- 2007–2012 had lower volume but relatively clean performance.
- From 2013–2016, both delinquency and charge-off rates increased significantly. 
- "Default" status is rare in the data. Most serious outcomes appear as "Charged Off".
- Late payments grew substantially after 2013. Could be due to increasing borrower stress after 2008 recession, or Lending Club's willingness to take on riskier loans. 

##### Some questions I'm considering:

- Why are outright "Defaults" rare, while "Charged Off" numbers are much higher? What does this pattern signal about their business practices, collections, and underwriting strategy?
- What does the steady rise in late payment statuses imply about borrower behavior and portfolio health?
- What incentives might Lending Club have had for allowing more loans to reach "Charged Off" status rather than earlier default?

In [ ]:
from scipy.stats import chi2_contingency, pearsonr, chi2
categorical_columns = ['addr_state', 'home_ownership', 'purpose', 'sub_grade', 'verification_status']

a = 0.05

for col in categorical_columns:
    p_contingency_table = pd.crosstab(dfa_data[col], dfa_data['delinquency_tf'])
    p_chi_sq, p_pval, p_dof, p_exp = chi2_contingency(p_contingency_table)
    p_crit_val = chi2.ppf((1-a), p_dof)
    print(f"Delinquent Population ---- {col}:------- \nchi2 = {p_chi_sq:.2f} \ncritical_value = {p_crit_val:.2f} \nsignificance ratio = {p_chi_sq / p_crit_val:.2f} \np-value = {p_pval} \ndof = {p_dof} \n\n")

for col in categorical_columns:
    p_contingency_table = pd.crosstab(dfa_data[col], dfa_data['defaulted_tf'])
    p_chi_sq, p_pval, p_dof, p_exp = chi2_contingency(p_contingency_table)
    p_crit_val = chi2.ppf((1-a), p_dof)
    print(f"Default Population ---- {col}:------- \nchi2 = {p_chi_sq:.2f} \ncritical_value = {p_crit_val:.2f} \nsignificance ratio = {p_chi_sq / p_crit_val:.2f} \np-value = {p_pval} \ndof = {p_dof} \n\n")


for col in categorical_columns:
    s_contingency_table = pd.crosstab(dfa[col], dfa['delinquency_tf'])
    s_chi_sq, s_pval, s_dof, s_exp = chi2_contingency(s_contingency_table)
    s_crit_val = chi2.ppf((1-a), s_dof)
    print(f"Delinquent Sample ---- {col}:------- \nchi2 = {s_chi_sq:.2f} \ncritical_value = {s_crit_val:.2f} \nsignificance ratio = {s_chi_sq / s_crit_val:.2f} \np-value = {s_pval} \ndof = {s_dof} \n\n")

for col in categorical_columns:
    s_contingency_table = pd.crosstab(dfa[col], dfa['defaulted_tf'])
    s_chi_sq, s_pval, s_dof, s_exp = chi2_contingency(s_contingency_table)
    s_crit_val = chi2.ppf((1-a), s_dof)
    print(f"Default Sample ---- {col}:------- \nchi2 = {s_chi_sq:.2f} \ncritical_value = {s_crit_val:.2f} \nsignificance ratio = {s_chi_sq / s_crit_val:.2f} \np-value = {s_pval} \ndof = {s_dof} \n\n")



### Chi-Square Test Results & Quick Observations

I ran Chi-square tests on several categorical variables to see how strongly they’re associated with `delinquency_tf`.

When I tested on the full dataset, almost every p-value came back as 0.0. That’s technically good news, but it doesn’t help me figure out which variables are actually the *strongest*. So I ran the tests on both the full population and different sample sizes to get a better sense of relative strength.

#### What is the Significance Ratio?

I’m still building my intuition around chi-square results. After some side research, I realized there isn’t one universal cutoff people use. So I started calculating a simple **Significance Ratio** to help me compare:

**Significance Ratio = chi² statistic ÷ critical value**

It should indicate **how many times stronger** the observed relationship is compared to what we’d expect if the variables were completely unrelated (pure chance).

**Example:**
- Delinquent sample `sub_grade` had a chi² of 32,629 and a critical value around 66.34 → Significance Ratio ≈ **671x**
- Delinquent population `sub_grade` is even higher with chi² of 122,867 and a critical value around 66.34 → Significance Ratio ≈ **2528x**

That’s an extremely strong signal.

#### Quick Observations:
- All the variables I tested are statistically significant.
- **`sub_grade`** is by far the strongest predictor — no surprise since it’s Lending Club’s own risk rating.
- I’m keeping an eye on overfitting risk, especially with high-cardinality variables like `addr_state` and `purpose`.

This step is helping me prioritize which categorical variables are actually worth keeping for the modeling phase.

In [ ]:
numeric_cols = ['annual_inc', 'delinq_2yrs', 'emp_length', 'emp_length_lt1','inq_last_6mths','loan_amnt', 'pub_rec', 'revol_bal']
adjusted_numerics = ['dti', 'fico_range_high', 'fico_range_low', 'int_rate', 'revol_util', 'term']

for col in adjusted_numerics:
    dfa_data[col + '_clean'] = dfa_data[col]
    dfa[col + '_clean'] = dfa[col]

p_corr_defaults = []
p_corr_delinquencies = []
p_adj_defaults = []
p_adj_delinquencies = []

s_corr_defaults = []
s_corr_delinquencies = []
s_adj_defaults = []
s_adj_delinquencies = []

for col in numeric_cols:
    p_1corr, p1_val = pearsonr(dfa_data[col], dfa_data['defaulted_tf'])
    p_corr_defaults.append({'variable': col,'correlation': p_1corr, 'pop_p_value': p1_val})

for col in numeric_cols:
    p_2corr, p2_val = pearsonr(dfa_data[col], dfa_data['delinquency_tf'])
    p_corr_delinquencies.append({'variable': col,'correlation': p_2corr, 'pop_p_value': p2_val})

for col in adjusted_numerics:
    temp = dfa_data[[col + '_clean', 'defaulted_tf']].dropna()
    p_3corr, p3_val = pearsonr(temp[col + '_clean'], temp['defaulted_tf'])
    p_adj_defaults.append({'variable': col, 'correlation': p_3corr, 'pop_p_value': p3_val})

    temp = dfa_data[[col + '_clean', 'delinquency_tf']].dropna()
    p_4corr, p4_val = pearsonr(temp[col + '_clean'], temp['delinquency_tf'])
    p_adj_delinquencies.append({'variable': col, 'correlation': p_4corr, 'pop_p_value': p4_val})



for col in numeric_cols:
    s_1corr, s1_val = pearsonr(dfa[col], dfa['defaulted_tf'])
    s_corr_defaults.append({'variable': col,'correlation': s_1corr, 'sample_p_value': s1_val})

for col in numeric_cols:
    s_2corr, s2_val = pearsonr(dfa[col], dfa['delinquency_tf'])
    s_corr_delinquencies.append({'variable': col,'correlation': s_2corr, 'sample_p_value': s2_val})

for col in adjusted_numerics:
    temp = dfa[[col + '_clean', 'defaulted_tf']].dropna()
    s_3corr, s3_val = pearsonr(temp[col + '_clean'], temp['defaulted_tf'])
    s_adj_defaults.append({'variable': col, 'correlation': s_3corr, 'sample_p_value': s3_val})

    temp = dfa[[col + '_clean', 'delinquency_tf']].dropna()
    s_4corr, s4_val = pearsonr(temp[col + '_clean'], temp['delinquency_tf'])
    s_adj_delinquencies.append({'variable': col, 'correlation': s_4corr, 'sample_p_value': s4_val})



p_corr_dflt = pd.DataFrame(p_corr_defaults).sort_values('correlation', ascending=False)
p_corr_dlqcy = pd.DataFrame(p_corr_delinquencies).sort_values('correlation', ascending=False)
p_adj_dflt = pd.DataFrame(p_adj_defaults).sort_values('correlation', ascending=False)
p_adj_dlqcy = pd.DataFrame(p_adj_delinquencies).sort_values('correlation', ascending=False)


s_corr_dflt = pd.DataFrame(s_corr_defaults).sort_values('correlation', ascending=False)
s_corr_dlqcy = pd.DataFrame(s_corr_delinquencies).sort_values('correlation', ascending=False)
s_adj_dflt = pd.DataFrame(s_adj_defaults).sort_values('correlation', ascending=False)
s_adj_dlqcy = pd.DataFrame(s_adj_delinquencies).sort_values('correlation', ascending=False)


print(f"POPULATION DEFAULTS:\n{p_corr_dflt}\n\n")
print(f"POPULATION DELINQUENCY:\n{p_corr_dlqcy}\n\n")
print(f"POPULATION DEFAULTS (Adjusted Columns):\n{p_adj_dflt}\n\n")
print(f"POPULATION DELINQUENCY (Adjusted Columns):\n{p_adj_dlqcy}\n\n")

print(f"SAMPLE DEFAULTS:\n{s_corr_dflt}\n\n")
print(f"SAMPLE DELINQUENCY:\n{s_corr_dlqcy}\n\n")
print(f"SAMPLE DEFAULTS (Adjusted Columns):\n{s_adj_dflt}\n\n")
print(f"SAMPLE DELINQUENCY (Adjusted Columns):\n{s_adj_dlqcy}\n\n")

### Thoughts and Notes

I decided to rerun the chi-square tests and pearson correlations for both the full population and the 500k sample. These notes should help me get closer to thoughtful feature selection for the modeling phase. Here’s where my head is at right now:

I had to separate the numeric columns into two buckets. The adjusted numerics are columns that contain NaNs. 

**Strongest signals so far:**
- `sub_grade` – Clearly the strongest categorical predictor. Makes sense since it's literally designed to reflect risk based on borrower history and patterns.
- `inq_last_6mths` – Strong positive correlation with default/delinquency. Recent credit inquiries are a classic risk signal.
- `int_rate` – Once I properly handled the nulls, it showed up as one of the strongest numeric predictors.

**Next steps and thoughts I’m considering:**
- I’m planning to rule out `dti`, `annual_inc`, `revol_bal`, and `revol_util` for the main ML model — they’re just too weak.
- `int_rate` is statistically strong, but I’m concerned it might be partly post-hoc. Since it’s based on Lending Club’s own risk assessment (grade + other factors), using it could make the model overfit or copy their past decisions instead of finding independent risk signals. Especially since Lending Club ultimately failed and had to restructure. "Leakage" could also end up affecting my model. 
- It’ll probably be worth training a few different model versions with and without `int_rate`, to see how much it actually helps or hurts real predictive value.
- Also remembering that different years had different dynamics. The current sample I’m working with is 2014–2017.
- Potential next step is testing how strongly certain variables correlate with each other in pairs. (Like FICO high vs low)



In [ ]:
numeric_cols = ['annual_inc', 'delinq_2yrs', 'emp_length', 'emp_length_lt1','inq_last_6mths','loan_amnt', 'pub_rec', 'revol_bal']
adjusted_numerics = ['dti', 'fico_range_high', 'fico_range_low', 'int_rate', 'revol_util', 'term']

p_numeric_comparisons = []
p_adjusted_numeric_comparisons = []
s_numeric_comparisons = []
s_adjusted_numeric_comparisons = []


for i in range(len(numeric_cols)):
    for j in range(i + 1, len(numeric_cols)):
        col1 = numeric_cols[i]
        col2 = numeric_cols[j]
        corr, p = pearsonr(dfa_data[col1], dfa_data[col2])
        p_numeric_comparisons.append({'comparison': f"{col1} VS {col2}", 'correlation': corr, 'p-value': f"{p:.4e}"})
        p_nums = pd.DataFrame(p_numeric_comparisons).sort_values('correlation', ascending=False)

for i in range(len(adjusted_numerics)):
    for j in range(i + 1, len(adjusted_numerics)):
        col1 = adjusted_numerics[i]
        col2 = adjusted_numerics[j]
        temp = dfa_data[[col1, col2]].dropna()
        temp_vs_dfa_data_size = 1 - (len(temp) / len(dfa_data))
        corr, p = pearsonr(temp[col1], temp[col2])
        p_adjusted_numeric_comparisons.append({'comparison': f"{col1} VS {col2}", 'correlation': corr, 'p-value': f"{p:.4e}", 'reduction': temp_vs_dfa_data_size})
        p_adj_nums = pd.DataFrame(p_adjusted_numeric_comparisons).sort_values('correlation', ascending=False)



# Sample
for i in range(len(numeric_cols)):   
    for j in range(i + 1, len(numeric_cols)): 
        col1 = numeric_cols[i]   
        col2 = numeric_cols[j]   
        corr, p = pearsonr(dfa[col1], dfa[col2])   
        s_numeric_comparisons.append({'comparison': f"{col1} VS {col2}", 'correlation': corr,'p-value': f"{p:.4e}"})   
        s_nums = pd.DataFrame(s_numeric_comparisons).sort_values('correlation', ascending=False)   


# Sample
for i in range(len(adjusted_numerics)): 
    for j in range(i + 1, len(adjusted_numerics)):   
        col1 = adjusted_numerics[i]
        col2 = adjusted_numerics[j]
        temp = dfa[[col1, col2]].dropna()
        temp_vs_dfa = 1 - len(temp) / len(dfa)
        corr, p = pearsonr(temp[col1], temp[col2])
        s_adjusted_numeric_comparisons.append({'comparison': f"{col1} VS {col2}", 'correlation': corr,'p-value': f"{p:.4e}", 'reduction': temp_vs_dfa})
        s_adj_nums = pd.DataFrame(s_adjusted_numeric_comparisons).sort_values('correlation', ascending=False)   



display(p_adj_nums.iloc[1:])
display(p_nums.iloc[:])
display(s_adj_nums.iloc[1:])
display(s_nums.iloc[:])


### Pairwise Feature Correlation Analysis (Multicollinearity Check)

After finishing the univariate analysis, I decided to examine how the numeric variables related to *each other* before moving into modeling. I intuitively recognized this as an important next step for good feature selection, even though I didn’t know the formal name for it at the time (pairwise correlation analysis to check for multicollinearity).

I wanted to be deliberate rather than just throwing every variable into an algorithm. I’m focused on building repeatable, thoughtful steps I can use in future projects.

**Why this step helps:**
- Prevents redundant features that can destabilize or reduce interpretability of the model
- Helps identify the most independent, useful signals
- Improves overall model quality and reduces risk of overfitting due to correlated predictors

### Note on Next Steps
The following section (Variance Inflation Factor and model-based feature importance) is completely new territory for me. I have not covered multicollinearity diagnostics or early model feature ranking in the Codecademy course. I'm exploring these techniques here to better validate my variable selection before moving into full modeling. I also want to record repeatable steps for quality checks in any future projects. 


In [ ]:
#Calculate VIF:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import numpy as np

# List your numeric features
numeric_features = ['annual_inc', 'delinq_2yrs', 'dti', 'emp_length', 
                    'fico_range_high', 'fico_range_low', 'inq_last_6mths',
                    'int_rate', 'loan_amnt', 'pub_rec', 'revol_bal',
                    'revol_util', 'term']

# Create a clean numeric version (force regular float64 + drop NaNs)
X = dfa_data[numeric_features].copy()

# Convert to regular float64 and drop rows with any NaN
X = X.astype('float64').dropna()

print(f"Rows used for VIF calculation: {len(X):,}")

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                   for i in range(len(X.columns))]

vif_data = vif_data.sort_values("VIF", ascending=False).round(2)

print("\nVIF Scores:")
print(vif_data)

X_sample = dfa[numeric_features].copy()
X_sample = X_sample.astype('float64').dropna()

print(f"Rows used for sample VIF: {len(X_sample):,}")

# Calculate VIF
vif_sample = pd.DataFrame()
vif_sample["feature"] = X_sample.columns
vif_sample["VIF"] = [variance_inflation_factor(X_sample.values, i) 
                     for i in range(len(X_sample.columns))]

vif_sample = vif_sample.sort_values("VIF", ascending=False).round(2)

print("\nVIF Scores - SAMPLE:")
print(vif_sample)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Features to use
features = ['annual_inc', 'delinq_2yrs', 'dti', 'emp_length', 'emp_length_lt1',
            'fico_range_high', 'fico_range_low', 'inq_last_6mths',
            'int_rate', 'loan_amnt', 'pub_rec', 'revol_bal',
            'revol_util', 'term', 'sub_grade', 'home_ownership', 
            'purpose', 'verification_status']

X = dfa_data[features].copy()
y = dfa_data['defaulted_tf']

# Convert categoricals to numeric (simple encoding for quick check)
X = pd.get_dummies(X, drop_first=True)

# Train quick Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

# Get feature importance
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Random Forest Feature Importance (Top 15):")
print(importance.head(15).round(4))

In [ ]:
"""
Importance Score Meaning
> 0.10 -- Very strong predictor
0.05 - 0.10 -- Strong / meaningful
0.01 - 0.05 -- Moderate contribution
< 0.01 -- Weak / probably not useful
"""


### Conclusion – File 2: Exploratory Analysis

This phase gave me a much clearer picture of the data and helped me start making thoughtful feature selection decisions.

**Key Takeaways:**
- Sub-grade, recent credit inquiries, and debt-to-income levels consistently emerged as the strongest signals across multiple methods.
- FICO range variables are highly redundant — I kept only the lower bound for a more conservative risk view.
- Interest rate is statistically strong but carries potential leakage risk, so I plan to test models both with and without it.
- Lending Club’s portfolio showed clear growth after 2013, with delinquency rates peaking mid-decade before declining — suggesting changes in underwriting standards over time.

I now have a shortlist of the most promising predictors and a better understanding of the business dynamics behind the data. 

Next: Build and compare a few Random Forest models using the selected features.

#### Other notes:

##### FICO Ranges
After reviewing VIF and Random Forest importance, I decided to keep only `fico_range_low`. 

My basic inclination is that a more "pessimistic" variable for a risk evaluation model is probably sound. At least at face value. 

##### Interest Rate
So far - I feel like it will be a good move to create separate models that include or exclude interest rates. There is a risk for leakage. My sense is that we still need to test one very strong variable and see what happens. 

#### Note
- Missing dti values only occur when annual_inc = 0. Filling them with 0 would be misleading. 
- Creating a separate dti_missing flag feels like it could complicate the analysis and create extra branches I have to manage. At the same time, I don’t want to drop the rows or exclude dti from testing entirely.

### My Judgement Calls & Independent Decisions so far

Throughout this project, I’ve made several deliberate decisions that differed from AI recommendations. I’m documenting them here to track my own reasoning and show the thinking process behind the work. I also want to keep this list on hand for any future work.

**1. Handling Missing `dti` Values**  
AI initially leaned toward dtype conversion (Float64 → float64) as the main fix for correlation issues.  
- I pushed back and argued the real problem was how we handled the nulls, not just the nullable dtype.  
- **Outcome**: I was right. The successful fix came from using temporary DataFrames with `.dropna()` on just the two columns being correlated. This preserved the original data while giving us real correlation values. The dtype change alone didn’t solve it.

**2. Sample Size Strategy**  
AI repeatedly suggested working primarily with smaller samples for speed and memory reasons.  
- I insisted on keeping the full `dfa_data` for core analysis and only using samples for validation/testing. I wanted accurate benchmarks, not just fast ones.  
- This led to cleaner, more reliable results across the board.

**3. Variable Definitions & Target Flexibility**  
- I chose to maintain **both** `delinquency_tf` (broader late/charge-off) and `defaulted_tf` (stricter) instead of collapsing them early.  
- This gives much more analytical flexibility when looking at severity levels.

**4. Non-Destructive Data Handling**  
- I consistently pushed to avoid inplace modifications or global drops/fills.  
- This led to the clean `temp = ... .dropna()` pattern we used for correlations, which protected data integrity.

#### Where I'm at now:
I'm trying to balance statistical cleanliness with my intuition on real world interpretations. I need to weigh whether excluding the <NA> dti rows will be valuable for more tests, and modeling, or if it's better to keep the full data set intact. 

In [ ]:
# === DTI Investigation ===

print("=== DTI Data Quality Check ===")

# 1. How many missing values?
print(f"Missing dti values: {dfa_data['dti'].isna().sum():,}")
print(f"Percentage missing: {dfa_data['dti'].isna().mean()*100:.2f}%")

# 2. Data type
print(f"dti data type: {dfa_data['dti'].dtype}")

# 3. Sample of missing dti rows
print("\n------------------------DTI <NA>/NaN sample set----------------------------\n")
print(dfa_data[dfa_data['dti'].isna()].head(20)[['loan_amnt', 'annual_inc', 'dti', 'defaulted_tf','delinquency_tf', 'loan_status']])
print("\n----------------------------------------------------\n")
# 4. Default rate when dti is missing
missing_default_rate = dfa_data[dfa_data['dti'].isna()]['defaulted_tf'].mean()
missing_delinquent_rate = dfa_data[dfa_data['dti'].isna()]['delinquency_tf'].mean()
print(f"\nDefault rate of DTI <NA>/NaN: {missing_default_rate:.4f} ({missing_default_rate*100:.2f}%)")
print(f"\nDelinquent rate of DTI <NA>/NaN: {missing_delinquent_rate:.4f} ({missing_delinquent_rate*100:.2f}%)\n\n")

In [ ]:
plt.figure(figsize=(10, 6))
dfa['loan_status'].value_counts().plot(kind='bar')

plt.title('Loan Status Distribution')
plt.xlabel('Loan Status')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
print("=== DTI Detailed Check ===")

# 1. Data type
print(f"dti dtype: {dfa_data['dti'].dtype}")

# 2. Number of missing values
print(f"Missing (NaN) values: {dfa_data['dti'].isna().sum():,}")

# 3. Number of <NA> (pandas nullable)
print(f"<NA> values: {dfa_data['dti'].isna().sum():,}")   # same check

# 4. Sample of dti values
print("\nFirst 10 dti values:")
print(dfa_data['dti'].head(10))

# 5. Any non-numeric values?
print("\nUnique dti types:")
print(dfa_data['dti'].apply(type).value_counts())